## Backtest statistical significance

 That strategy buys (sells) the TU future if it has a positive (negative) 12-month return, and holds the position for 1 month. 

In [43]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [6]:
df = yf.download("ZF=F", start="2000-01-01")
df.columns = df.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed


In [7]:
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2000-09-21,100.421898,100.468803,100.171898,100.187500,10742
2000-09-22,100.281303,100.328102,100.281303,100.296898,6092
2000-09-25,100.296898,100.328102,100.296898,100.296898,6359
2000-09-26,100.484398,100.500000,100.421898,100.437500,8689
2000-09-27,100.468803,100.484398,100.453102,100.468803,12359


In [33]:
monthly = df["Close"].resample("ME").last()
monthly = monthly.to_frame("close")
monthly.index.name = "date"

In [34]:
monthly.head()

,close
date,
2000-09-30,100.593803
2000-10-31,100.718803
2000-11-30,101.937500
2000-12-31,103.578102
2001-01-31,104.234398


In [35]:
# 12 month trailing return
monthly["ret_12m"] = monthly["close"].pct_change(12)

# signal
monthly["signal"] = np.sign(monthly["ret_12m"])

# 1-month forward return
monthly["ret_1m_fwd"] = monthly["close"].pct_change(1).shift(-1)

# strategy return
monthly["strat_ret"] = monthly['signal'] * monthly['ret_1m_fwd']

In [37]:
monthly.dropna(inplace=True)
print(monthly.shape)
monthly[['ret_12m','signal','strat_ret']].head()

(300, 5)


,ret_12m,signal,strat_ret
date,,,
2001-09-30,0.075334,1.0,0.017189
2001-10-31,0.092460,1.0,-0.017467
2001-11-30,0.060546,1.0,-0.020956
2001-12-31,0.021874,1.0,0.002214
2002-01-31,0.017688,1.0,0.011195


### Summary Stats

In [40]:
strat = monthly["strat_ret"]
n = len(strat)
mean_m = strat.mean()
std_m = strat.std(ddof=1)
sharpe_m = mean_m / std_m
sharpe_ann = sharpe_m * np.sqrt(12)

In [42]:
print(f"n months        : {n}")
print(f"mean (monthly)  : {mean_m:.5f}")
print(f"std  (monthly)  : {std_m:.5f}")
print(f"Sharpe (monthly): {sharpe_m:.4f}")
print(f"Sharpe (annualized, naive): {sharpe_ann:.4f}")

print(strat.autocorr(lag=1))

n months        : 300
mean (monthly)  : 0.00093
std  (monthly)  : 0.01093
Sharpe (monthly): 0.0853
Sharpe (annualized, naive): 0.2953
0.06321842098183354


### Guassian (parametric) t-test

- H0: μ = 0 — the true mean monthly return of the strategy is zero; any positive average observed is just noise.
- H1: μ > 0 — the true mean monthly return is positive, consistent with a genuine time-series momentum premium.

In [44]:
t_stat = mean_m / ( std_m / np.sqrt(n))
p_value_gaussian = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))

print(f"t-stat        : {t_stat:.4f}")
print(f"p-value (2-sided): {p_value_gaussian:.4f}")

t-stat        : 1.4766
p-value (2-sided): 0.1408


In [46]:
alpha = 0.05
t_crit_two_sided = stats.t.ppf(1 - alpha/2, df=n-1)
t_crit_one_sided = stats.t.ppf(1 - alpha, df=n-1)

print(f"t-critical (two-sided, α=0.05): ±{t_crit_two_sided:.4f}")
print(f"t-critical (one-sided, α=0.05): {t_crit_one_sided:.4f}")
print(f"t-stat: {t_stat:.4f}")

t-critical (two-sided, α=0.05): ±1.9679
t-critical (one-sided, α=0.05): 1.6500
t-stat: 1.4766


**conclusion** \
t-stat (1.4766) < t-critical (1.6500) at α = 0.05 → fail to reject H0.

In plain terms: under the (naive, i.i.d.-normal) Gaussian test, there isn't enough statistical evidence to conclude this strategy's average return is genuinely positive rather than zero. The observed mean of 0.093%/month could plausibly be noise given the sample's variance and size (n=300). This doesn't prove the strategy has no edge — it means this particular test, under these assumptions, can't distinguish the edge from luck.

### Monte Carlo

- H0: Monthly price returns are i.i.d. draws from the same empirical distribution as the real data (same mean, vol, skew, kurtosis) — no genuine serial dependence. Any apparent profitability of the trend-following rule is an artifact of applying a fixed rule to any series with this mean/vol, not evidence of real momentum.
- H1: Real price paths have serial dependence that this rule genuinely exploits — the true momentum signal is not just noise dressed up as trend.

In [58]:
price_ret = monthly["close"].pct_change().dropna()
n_needed = len(monthly)

def run_strategy_on_prices(prices):
    s = pd.Series(prices)
    ret_12m = s.pct_change(12)
    signal = np.sign(ret_12m)
    ret_1m_fwd = s.pct_change(1).shift(-1)
    strat_ret = signal * ret_1m_fwd
    return strat_ret.dropna().mean()

np.random.seed(42)
n_sims = 5000

sim_means = np.empty(n_sims)
for i in range(n_sims):
    sim_returns = np.random.choice(price_ret.values, size=n_needed, replace=True)
    sim_prices = 100 * np.cumprod(1 + sim_returns)
    sim_means[i] = run_strategy_on_prices(sim_prices)

In [59]:
p_value_price_mc = (sim_means >= observed_mean).mean()
print(f"p-value (price-path Monte Carlo): {p_value_price_mc:.4f}")

p-value (price-path Monte Carlo): 0.0776


**Conclusion:**\
p = 0.078. At the conventional α = 0.05 threshold, fail to reject H0 (though it's closer to the boundary than the block bootstrap). Roughly 7.9% of purely random, no-momentum price paths with the same statistical moments as your real data would have produced a strategy return this good or better by chance alone. That's not overwhelming evidence of noise, but it's not strong evidence of a real effect either — this sits in the "suggestive but not conclusive" zone that a lot of published backtests conveniently stop reporting at.